# Story C: Behavioral Weirdness — Anomaly Detection Evaluation

**Purpose**: Evaluate and compare the anomaly detection models (Isolation Forest vs. LOF) for users, and analyze the distribution of movie polarization scores.
**Inputs**: `artifacts/story_C/tables/user_anomaly_scores.parquet`, `artifacts/story_C/tables/movie_anomaly_scores.parquet`
**Key Questions**:
1. How well do Isolation Forest (IF) and Local Outlier Factor (LOF) agree on who is an outlier?
2. What is the distribution of the combined anomaly score?
3. Are the polarizing movies truly controversial (high standard deviation vs. mean)?

In [ ]:
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# --- Paths ---
STORY_C_DIR = os.path.join('artifacts', 'story_C')
TABLES_DIR  = os.path.join(STORY_C_DIR, 'tables')
FIGURES_DIR = os.path.join(STORY_C_DIR, 'figures')

os.makedirs(FIGURES_DIR, exist_ok=True)
print("Setup OK")

## 1. Load Anomaly Results

In [ ]:
user_scores = pd.read_parquet(os.path.join(TABLES_DIR, 'user_anomaly_scores.parquet'))
movie_scores = pd.read_parquet(os.path.join(TABLES_DIR, 'movie_anomaly_scores.parquet'))

print(f"Users analyzed: {len(user_scores)}")
print(f"Movies analyzed: {len(movie_scores)}")
user_scores.head()

## 2. Model Agreement: IF vs. LOF

Since we don't have ground truth labels (we don't know who is *truly* a bot or a weird user), we evaluate by **consistency** between different unsupervised algorithms.

In [ ]:
# Correlation between raw scores
correlation = user_scores['iso_forest_score'].corr(user_scores['lof_score'])

plt.figure(figsize=(8, 6))
sns.regplot(data=user_scores.sample(min(2000, len(user_scores))), 
            x='iso_forest_score', y='lof_score', 
            scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.title(f"Score Correlation: {correlation:.3f}")
plt.xlabel("Isolation Forest Score (Lower is more anomalous in sklearn, but here pre-inverted)")
plt.ylabel("LOF Score")
plt.grid(True, alpha=0.3)
plt.show()

### Overlap Analysis
How many users are flagged as anomalies by BOTH models?

In [ ]:
if_anomalies = set(user_scores[user_scores['iso_forest_label'] == -1]['userId'])
lof_anomalies = set(user_scores[user_scores['lof_label'] == -1]['userId'])

intersection = if_anomalies.intersection(lof_anomalies)
union = if_anomalies.union(lof_anomalies)

print(f"IF Flags  : {len(if_anomalies)}")
print(f"LOF Flags : {len(lof_anomalies)}")
print(f"Both Flag : {len(intersection)}")
print(f"Jaccard Similarity: {len(intersection)/len(union):.3f}")

## 3. Score Distributions
Anomalies should be at the extreme right of the histogram.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(user_scores['combined_score'], bins=50, kde=True, ax=ax[0], color='purple')
ax[0].set_title("Combined Anomaly Score Distribution")

sns.boxplot(x=user_scores['combined_score'], ax=ax[1], color='lightgreen')
ax[1].set_title("Boxplot of Anomaly Scores")

plt.show()

## 4. Movie Polarization Evaluation
We visualize the relationship between Mean Rating and Rating Std Dev. Polarizing movies usually have high Std Dev regardless of the mean.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=movie_scores, x='rating_mean', y='rating_std', 
                hue='polarization_score', palette='viridis', alpha=0.6)
plt.title("Movie Mean vs. Std Dev (Color = Polarization Score)")
plt.xlabel("Average Rating")
plt.ylabel("Standard Deviation of Ratings")

# Label top 5 polarized movies
for i in range(5):
    row = movie_scores.iloc[i]
    plt.text(row['rating_mean']+0.05, row['rating_std'], row['title'][:20], 
             fontsize=9, fontweight='bold', color='darkred')

plt.show()

## 5. Metrics Summary Table

In [ ]:
eval_data = {
    "Total Users Checked": len(user_scores),
    "IF Anomaly Count": len(if_anomalies),
    "LOF Anomaly Count": len(lof_anomalies),
    "Mutual Agreement Count": len(intersection),
    "Agreement % (of total flagging)": round(len(intersection)/len(union)*100, 2),
    "Mean Anomaly Score (Combined)": round(user_scores['combined_score'].mean(), 4),
    "Max Polarization Score": round(movie_scores['polarization_score'].max(), 4)
}

df_eval = pd.DataFrame(eval_data.items(), columns=['Metric', 'Value'])
df_eval.to_csv(os.path.join(STORY_C_DIR, 'reports', 'eval_anomaly_metrics.csv'), index=False)
df_eval

## 6. Interpretation

1. **Model Reliability**: If the agreement between IF and LOF is high (> 30% Jaccard), we can be confident that these users exhibit multivariate outliers. If they disagree, it means IF is catching global outliers (density spread) while LOF is catching local density deviations.
2. **Polarization**: Movies with higher standard deviation (> 1.2) are typical candidates for 'Love it or Hate it' cinema. The robust z-score helps normalize this across movies with different sample sizes.
3. **Next Steps**: Inspect the `case_studies.md` report for detailed profiles of the top flagged users to see if they are 'power users' or potential bots (non-human entropy in rating patterns).